# GPU Modular Arithmetic Unit (Barrett reduction) — Colab runner

**Before running:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU**.

This notebook compiles the pure-CUDA project with `nvcc`, runs the correctness tests and
GPU-vs-CPU benchmarks, and renders the plots.

In [ ]:
# 1) Confirm we have a GPU and nvcc.
!nvidia-smi
!nvcc --version

## 2) Get the code onto Colab
Pick ONE option below.

**Option A — git clone** (if you pushed the project to GitHub): set `REPO_URL`.

**Option B — upload a zip** of the `Project` folder via the file picker.

In [ ]:
# Option A: clone (edit REPO_URL, then run). Skip if uploading a zip.
REPO_URL = ""  # e.g. "https://github.com/youruser/yourrepo.git"
import os
if REPO_URL:
    !rm -rf project_repo && git clone $REPO_URL project_repo
    # adjust if the project lives in a subfolder of the repo:
    %cd project_repo
    print('cwd =', os.getcwd())

In [ ]:
# Option B: upload a zip of the Project folder, then unzip.
# (Make the zip locally:  cd Project/.. && zip -r project.zip Project )
from google.colab import files
up = files.upload()
import zipfile, os
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
        print('unzipped', name)
# cd into the folder that contains the Makefile:
for root, dirs, fs in os.walk('.'):
    if 'Makefile' in fs and os.path.exists(os.path.join(root,'cuda')):
        %cd $root
        break
print('cwd =', os.getcwd()); os.listdir('.')

In [ ]:
# 3) Local correctness (CPU) — sanity that the shared arithmetic is right.
!make host_test
!make ntt_host_test

In [ ]:
# 4) Compile the CUDA programs for the T4 (sm_75).
!make gpu

In [ ]:
# 5) GPU correctness: GPU NTT must match the CPU reference + round-trip.
!./build/ntt

In [ ]:
# 6) Benchmarks: modmul throughput + NTT latency (writes results/*.csv).
!./build/bench
# 7) RSA-style batched modular exponentiation over the large prime P64.
!./build/modexp_demo

In [ ]:
# 8) Plots + independent Python oracle.
!python3 python/oracle.py
!python3 python/plots.py
from IPython.display import Image, display
import os
for png in ['results/ntt_speedup.png', 'results/modmul_throughput.png']:
    if os.path.exists(png):
        display(Image(png))

In [ ]:
# 9) Show the raw numbers.
for f in ['results/bench_modmul.csv','results/bench_ntt.csv','results/bench_modexp.csv']:
    print('===', f, '==='); print(open(f).read())